# 01 — Extracción

**Fase 1 del proyecto ISO 20022 en Redes DLT.**

Se conecta a las APIs públicas de **XRPL**, **Stellar** y **Hedera** para extraer transacciones históricas con memo no vacío, y en paralelo descarga indicadores macroeconómicos de **FRED**. Los datos crudos se guardan en `data/raw/` (por red) y `data/external/` (FRED).

# 1. Configuración

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = True  # activa guardado de archivos
FILTRO_MEMO: bool = True

# 2. Imports

In [ ]:
import os
import json
import time
import pandas as pd
import pyarrow
import requests
import httpx
from datetime import datetime, date, timezone
from typing import Any
from google.cloud import bigquery
from google.colab import drive, userdata, auth

auth.authenticate_user()
drive.mount('/content/drive')

# 3. Paths y Constantes

In [ ]:
# Paths de salida
BASE_DIR: str = "/content/drive/MyDrive/Tesis/main"
DIR_RAW: dict[str, str] = {
    "xrpl": os.path.join(BASE_DIR, "data", "raw", "xrpl"),
    "stellar": os.path.join(BASE_DIR, "data", "raw", "stellar"),
    "hedera": os.path.join(BASE_DIR, "data", "raw", "hedera"),
    "algorand": os.path.join(BASE_DIR, "data", "raw", "algorand"),}
DIR_EXTERNAL: str = os.path.join(BASE_DIR, "data", "external")

In [ ]:
# Parámetros globales
RANGO_INICIO: str = "2026-01-01"
RANGO_FIN: str = date.today().isoformat()
GRANULARIDAD: str = "1D"

# Límites de seguridad para las llamadas paginadas
TIMEOUT: int = 30
MAX_PAGINAS: int = 50
PAGINA_STELLAR: int = 200
PAGINA_HEDERA: int = 100
PAGINA_ALGORAND: int = 1000
XRPL_LEDGER_INICIO: int = 101256700  # ~2026-01-01 00:00 UTC

# Series macroeconómicas de FRED a extraer
FRED_SERIES: dict[str, str] = {
    "STLFSI4": "St. Louis Fed Financial Stress Index",
    "DFF": "Federal Funds Effective Rate",
    "SOFR": "Secured Overnight Financing Rate",
    "RRPONTSYD": "Overnight Reverse Repurchase Agreements",}

In [ ]:
# Endpoints
STELLAR_HORIZON: str = "https://horizon.stellar.org"
HEDERA_MIRROR: str = "https://mainnet-public.mirrornode.hedera.com"
ALGORAND_INDEXER: str = "https://mainnet-idx.algonode.cloud"
FRED_API: str = ("https://api.stlouisfed.org/fred/series/observations")
FRED_API_KEY = userdata.get("FRED_API_KEY")

# BigQuery
PROJECT_ID = "tesis-xrpl"
cliente = bigquery.Client(project=PROJECT_ID)

# Test de conexión
print("TEST DE CONEXION A APIS")
print("Stellar:", requests.get(STELLAR_HORIZON, timeout=TIMEOUT).status_code)
print("Hedera: ", requests.get(HEDERA_MIRROR + "/api/v1/transactions?limit=1", timeout=TIMEOUT).status_code)
print("Algorand:", requests.get(ALGORAND_INDEXER, timeout=TIMEOUT).status_code)
print("FRED:", requests.get(FRED_API, params={"api_key": FRED_API_KEY, "series_id": "DFF", "file_type": "json"}).status_code)

# 4. Funciones

In [ ]:
def guardar_json(datos: Any, ruta: str) -> None:
    """
    Guarda una estructura como JSON si SAVE_DATA está activo.

    Args:
        datos: objeto serializable a JSON.
        ruta: path de destino del archivo.
    """
    if not SAVE_DATA:
        if DEBUG:
            print(f"[DEBUG] SAVE_DATA=False, no guardo {ruta}")
        return
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    if DEBUG:
        print(f"[DEBUG] guardado: {ruta}")

In [ ]:
def xrpl_extraer_bigquery(cliente, ledger_inicio: int) -> pd.DataFrame:
    """
    Extrae txs XRPL con memo desde BigQuery (YTD 2026).

    Args:
        cliente: bigquery.Client autenticado.
        ledger_inicio: LedgerIndex de arranque (corte YTD).

    Returns:
        DataFrame con transacciones exitosas y memo no vacío.
    """
    consulta = f"""
    SELECT
        `hash` AS tx_id,
        Account AS emisor,
        Destination AS receptor,
        AmountXRP AS amount,
        Fee AS fee,
        LedgerIndex AS ledger_index,
        memo.Memo.MemoData AS memo_raw
    FROM `xrpledgerdata.fullhistory.transactions`,
        UNNEST(Memos) AS memo
    WHERE LedgerIndex >= {ledger_inicio}
        AND TransactionResult = "tesSUCCESS"
        AND memo.Memo.MemoData IS NOT NULL
    """
    df = cliente.query(consulta).to_dataframe()
    if DEBUG:
        print(f"[DEBUG] XRPL BigQuery: {len(df)} txs con memo")
    return df

In [ ]:
def stellar_extraer(inicio: str,
                    max_paginas: int) -> list[dict]:
    """
    Pagina Horizon y recupera txs con memo hasta 'inicio'.

    Args:
        inicio: fecha mínima (ISO) de created_at a incluir.
        max_paginas: tope de páginas a recorrer.

    Returns:
        Lista de transacciones Stellar con memo no vacío.
    """
    url = (f"{STELLAR_HORIZON}/transactions"
           f"?order=desc&limit={PAGINA_STELLAR}"
           f"&include_failed=false")
    txs_memo: list[dict] = []
    with httpx.Client(timeout=TIMEOUT) as cliente:
        for _ in range(max_paginas):
            try:
                r = cliente.get(url)
                r.raise_for_status()
                datos = r.json()
            except (httpx.HTTPError, ValueError) as e:
                if DEBUG:
                    print(f"[DEBUG] error Stellar: {e}")
                break
            registros = datos.get(
                "_embedded", {}).get("records", [])
            if not registros:
                break
            for tx in registros:
                if (not FILTRO_MEMO
                        or tx.get("memo_type") != "none"):
                    txs_memo.append(tx)
            if registros[-1].get("created_at", "") < inicio:
                break
            url = datos.get(
                "_links", {}).get("next", {}).get("href", "")
            if not url:
                break
    if DEBUG:
        print(f"[DEBUG] Stellar: {len(txs_memo)} txs con memo")
    return txs_memo

In [ ]:
def hedera_extraer(inicio: str,
                   max_paginas: int) -> list[dict]:
    """
    Pagina el Mirror Node y recupera txs con memo.

    Args:
        inicio: fecha mínima (ISO) a incluir.
        max_paginas: tope de páginas a recorrer.

    Returns:
        Lista de transacciones Hedera con memo_base64.
    """
    ruta = (f"/api/v1/transactions?limit={PAGINA_HEDERA}"
            f"&order=desc")
    inicio_epoch = datetime.fromisoformat(inicio).replace(
        tzinfo=timezone.utc).timestamp()
    txs_memo: list[dict] = []
    with httpx.Client(timeout=TIMEOUT) as cliente:
        for _ in range(max_paginas):
            try:
                r = cliente.get(HEDERA_MIRROR + ruta)
                r.raise_for_status()
                datos = r.json()
            except (httpx.HTTPError, ValueError) as e:
                if DEBUG:
                    print(f"[DEBUG] error Hedera: {e}")
                break
            registros = datos.get("transactions", [])
            if not registros:
                break
            for tx in registros:
                if not FILTRO_MEMO or tx.get("memo_base64"):
                    txs_memo.append(tx)
            ultimo = registros[-1].get(
                "consensus_timestamp", "0")
            if float(ultimo) < inicio_epoch:
                break
            ruta = datos.get("links", {}).get("next", "")
            if not ruta:
                break
    if DEBUG:
        print(f"[DEBUG] Hedera: {len(txs_memo)} txs con memo")
    return txs_memo

In [ ]:
def algorand_extraer(inicio: str,
                     max_paginas: int) -> list[dict]:
    """
    Pagina el Indexer de Algorand y recupera txs con note.

    Args:
        inicio: fecha mínima (ISO) a incluir (after-time).
        max_paginas: tope de páginas a recorrer.

    Returns:
        Lista de transacciones Algorand con campo note.
    """
    url = (f"{ALGORAND_INDEXER}/v2/transactions"
           f"?after-time={inicio}&limit={PAGINA_ALGORAND}")
    txs_memo: list[dict] = []
    with httpx.Client(timeout=TIMEOUT) as cliente:
        for pagina in range(max_paginas):
            try:
                r = cliente.get(url)
                r.raise_for_status()
                datos = r.json()
            except (httpx.HTTPError, ValueError) as e:
                if DEBUG:
                    print(f"[DEBUG] error Algorand: {e}")
                break
            registros = datos.get("transactions", [])
            if not registros:
                break
            for tx in registros:
                if not FILTRO_MEMO or tx.get("note"):
                    txs_memo.append(tx)
            token = datos.get("next-token", "")
            if not token:
                break
            url = (f"{ALGORAND_INDEXER}/v2/transactions"
                   f"?after-time={inicio}"
                   f"&limit={PAGINA_ALGORAND}&next={token}")
            if DEBUG:
                print(f"[DEBUG] Algorand pág. {pagina + 1} "
                      f"- {len(txs_memo)} txs acumuladas")
    if DEBUG:
        print(f"[DEBUG] Algorand: {len(txs_memo)} txs con note")
    return txs_memo

In [ ]:
def fred_serie(series_id: str, inicio: str,
               fin: str) -> list[dict]:
    """
    Descarga las observaciones de una serie de FRED.

    Args:
        series_id: identificador de la serie (p. ej. 'DFF').
        inicio: fecha inicial (ISO).
        fin: fecha final (ISO).

    Returns:
        Lista de observaciones (fecha/valor), o [] si falla.
    """
    if not FRED_API_KEY:
        if DEBUG:
            print("[DEBUG] falta FRED_API_KEY, omito FRED")
        return []
    params = {
        "series_id": series_id,
        "observation_start": inicio,
        "observation_end": fin,
        "api_key": FRED_API_KEY,
        "file_type": "json",
    }
    try:
        r = requests.get(FRED_API, params=params,
                         timeout=TIMEOUT)
        r.raise_for_status()
        return r.json().get("observations", [])
    except (requests.RequestException, ValueError) as e:
        if DEBUG:
            print(f"[DEBUG] error FRED {series_id}: {e}")
        return []


def fred_extraer(inicio: str,
                 fin: str) -> dict[str, list[dict]]:
    """
    Descarga todas las series definidas en FRED_SERIES.

    Args:
        inicio: fecha inicial (ISO).
        fin: fecha final (ISO).

    Returns:
        Diccionario {series_id: observaciones}.
    """
    resultado: dict[str, list[dict]] = {}
    for series_id in FRED_SERIES:
        obs = fred_serie(series_id, inicio, fin)
        resultado[series_id] = obs
        if DEBUG:
            print(f"[DEBUG] FRED {series_id}: {len(obs)} obs")
    return resultado

# 5. Proceso

In [ ]:
# Sello temporal (UTC) para nombrar los archivos crudos
sello: str = datetime.now(timezone.utc).strftime("%Y%m%d")

stellar_txs = stellar_extraer(RANGO_INICIO, MAX_PAGINAS)
hedera_txs = hedera_extraer(RANGO_INICIO, MAX_PAGINAS)
algorand_txs = algorand_extraer(RANGO_INICIO, MAX_PAGINAS)
fred_datos = fred_extraer(RANGO_INICIO, RANGO_FIN)

if DEBUG:
    print("\n[DEBUG] Resumen de extracción")
    print(f"  Stellar:  {len(stellar_txs)} txs")
    print(f"  Hedera:   {len(hedera_txs)} txs")
    print(f"  Algorand: {len(algorand_txs)} txs")
    print(f"  FRED:     {len(fred_datos)} series")

# 6. Save

In [ ]:
if SAVE_DATA:
    guardar_json(stellar_txs, os.path.join(DIR_RAW["stellar"], f"stellar_{sello}.json"))
    guardar_json(hedera_txs, os.path.join(DIR_RAW["hedera"], f"hedera_{sello}.json"))
    guardar_json(algorand_txs, os.path.join(DIR_RAW["algorand"], f"algorand_{sello}.json"))
    guardar_json(fred_datos, os.path.join(DIR_EXTERNAL, f"fred_{sello}.json"))
    if DEBUG:
        print("[DEBUG] guardado completo")
elif DEBUG:
    print("[DEBUG] SAVE_DATA=False, datos solo en memoria")

# Backlog

> **Nota XRPL**: el RPC público no indexa por fecha. La extracción recorre un rango de ledgers recientes acotado por XRPL_MAX_LEDGERS (ajustable). Stellar y Hedera sí permiten cortar por fecha mediante paginación.